# CodePause Phase 4 — Dataset v4 + Laziness Detection Loop

**Goal:** train/evaluate adapter v4 after adding laziness detection.

**Final verified outcome:** FAIL on Colab T4.
- Base + `best_phase3_prompt`: `2/30`
- Adapter v4 + `best_phase3_prompt`: `0/30`
- Delta: `-2/30`

Keep this notebook as historical evidence. Do **not** reuse v4 as the forward path.

## 1. Setup

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = 'https://github.com/alesierraalta/AMD-Hacktaton-thinking-middle.git'
PROJECT_DIR = pathlib.Path('/content/AMDh')

if PROJECT_DIR.exists() and (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'reset', '--hard'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    raise RuntimeError(f'{PROJECT_DIR} exists but is not a git checkout')

os.chdir(PROJECT_DIR)
print('cwd:', pathlib.Path.cwd())
subprocess.run(['git', 'log', '--oneline', '-1'], check=True)

## 2. Dependencies + smoke tests

In [ ]:
import subprocess, sys
deps = ['trl', 'peft', 'accelerate', 'bitsandbytes', 'datasets', 'transformers', 'torchao>=0.16.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *deps], check=True)
subprocess.run(['python', '-m', 'pytest', 'tests/test_generate_sft_v4.py', 'tests/test_metrics.py', 'tests/test_evaluator.py', '-q'], check=True)

## 3. Generate and verify dataset v4

In [ ]:
import subprocess
subprocess.run(['python', 'scripts/generate_sft_v4.py'], check=True)
subprocess.run(['python', 'scripts/verify_v4_metrics.py'], check=True)


## 4. Baseline — base + best prompt

In [ ]:
!python eval/evaluate_baseline.py \
  --model_name Qwen/Qwen1.5-1.8B-Chat \
  --problems_path data/heldout_problems_30.jsonl \
  --output_path results/phase4_base_best_prompt.jsonl \
  --prompt_template best_phase3_prompt

## 5. Train adapter v4

In [ ]:
!python training/sft_lora.py \
  --model_name Qwen/Qwen1.5-1.8B-Chat \
  --dataset_path data/thinkanywhere_sft_v4.jsonl \
  --output_dir results/sft_phase4_v4 \
  --epochs 1 \
  --max_steps 150 \
  --max_seq_length 1024 \
  --device auto \
  --load_in_4bit \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 8 \
  --learning_rate 1e-4 \
  --lora_r 8 \
  --lora_alpha 16 \
  --lora_dropout 0.05

## 6. Evaluate adapter v4

In [ ]:
!python eval/evaluate_finetuned.py \
  --base_model Qwen/Qwen1.5-1.8B-Chat \
  --adapter_path results/sft_phase4_v4 \
  --problems_path data/heldout_problems_30.jsonl \
  --output_path results/phase4_adapter_v4_best_prompt.jsonl \
  --prompt_template best_phase3_prompt

## 7. Report + hard gate

In [ ]:
import json, pathlib

!python eval/make_report.py \
  --baseline results/phase4_base_best_prompt.jsonl \
  --finetuned results/phase4_adapter_v4_best_prompt.jsonl \
  --out results/phase4_report.md

def score(path):
    rows = [json.loads(x) for x in pathlib.Path(path).read_text().splitlines() if x.strip()]
    return sum(1 for r in rows if r.get('passed')), len(rows)

base_pass, base_total = score('results/phase4_base_best_prompt.jsonl')
adapter_pass, adapter_total = score('results/phase4_adapter_v4_best_prompt.jsonl')
print('base + best prompt:', f'{base_pass}/{base_total}')
print('adapter v4 + best prompt:', f'{adapter_pass}/{adapter_total}')
print('delta:', f'{adapter_pass - base_pass}/{base_total}')
print('HARD_GATE:', 'PASS' if adapter_pass > base_pass else 'FAIL')
